# Devoir 1 : Classification de sentiments de deux façons

Bienvenue à votre premier devoir ! Lorsque vous le soumettez sur ZoneCours, assurez-vous de renommer le fichier avec votre nom comme suit : **"Assignment1_{firstname}_{lastname}.ipynb"**. Soumettez une version du notebook où vous avez fait roulez toutes les cellules et laissez les sorties apparentes. La date de remise est fixée à 23 h 59 le 26 février. Le devoir est noté sur un total de 42 points, avec 2 points boni possibles. Si vous avez travaillé sur une quelconque partie du devoir avec d’autres personnes, veuillez indiquer leurs noms ici :

In [1]:
__author__ = "{Abdoul Wassi Badirou}"
__collaborators__ = "{Hilaire Touyem, Sandra Desmair Fogang Lontouo, Arthur Richel Dongmo Tsamo}"

Toutes les bibliothèques dont vous aurez besoin sont listées ici. Si certaines bibliothèques qui pourraient sembler pertinentes ont été exclues, c’est intentionnel afin que vous preniez le temps d’écrire vos propres fonctions. Veuillez ne pas modifier cette cellule et abstenez-vous d’ajouter d’autres bibliothèques ailleurs. Vous pouvez utiliser toute fonction que vous jugez utile provenant de ces bibliothèques.

In [2]:
from nltk.corpus import movie_reviews
import string
import re
import random
import math
from collections import Counter
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
random.seed(202601)
torch.manual_seed(202401)
device = torch.device('cpu')

Nous allons comparer deux types de classificateurs de critiques de films : (1) un classificateur de régression logistique basé sur des N-grammes et (2) un classificateur basé sur un LSTM. La première étape consiste à télécharger les données et à créer nos ensembles d’entraînement et de test. Il y a deux classes de critiques : positives et négatives.

In [3]:
review_classes = movie_reviews.categories()
pos_reviews = [(movie_reviews.raw(fileid), 'pos') for fileid in movie_reviews.fileids(categories=['pos'])]
neg_reviews = [(movie_reviews.raw(fileid), 'neg') for fileid in movie_reviews.fileids(categories=['neg'])]
train_dataset = pos_reviews[:800] + neg_reviews[:800]
test_dataset = pos_reviews[800:] + neg_reviews[800:]

In [4]:
train_dataset[0]

('films adapted from comic books have had plenty of success , whether they\'re about superheroes ( batman , superman , spawn ) , or geared toward kids ( casper ) or the arthouse crowd ( ghost world ) , but there\'s never really been a comic book like from hell before . \nfor starters , it was created by alan moore ( and eddie campbell ) , who brought the medium to a whole new level in the mid \'80s with a 12-part series called the watchmen . \nto say moore and campbell thoroughly researched the subject of jack the ripper would be like saying michael jackson is starting to look a little odd . \nthe book ( or " graphic novel , " if you will ) is over 500 pages long and includes nearly 30 more that consist of nothing but footnotes . \nin other words , don\'t dismiss this film because of its source . \nif you can get past the whole comic book thing , you might find another stumbling block in from hell\'s directors , albert and allen hughes . \ngetting the hughes brothers to direct this see

In [5]:
len(train_dataset)

1600

1. **[3 points]** L’étape suivante consiste à tokeniser les critiques. Vous devez maintenant écrire une fonction de prétraitement qui prend une critique et (1) supprime la ponctuation, (2) met tous les mots en minuscules et (3) les sépare en tokens en utilisant tout type d’espacement. Assurez-vous qu’il n’y a pas de tokens vides.

In [6]:
def preprocess(review:str):
    tokenized_review = []
    punctuations = list(string.punctuation)
    ## TO DO
    ### Suppression ponctuation
    punctuationsStr = "".join(punctuations)
    pattern = f"[{re.escape(punctuationsStr)}]+"
    reviewPrepared = re.sub(pattern,'',review)
    ### Cast en minuscules
    reviewPrepared= reviewPrepared.lower()
    ### Séparation en token
    tokenized_review = reviewPrepared.split()

    if "" in tokenized_review:
        raise ValueError("ATTENTION: tokenized_review contient des tokens vides") 

    
    ##
    return tokenized_review

## Classificateur 1 : Caractéristiques N‑grammes et régression logistique

Ce premier classificateur extrait des caractéristiques de type N‑grammes à partir des critiques, les transforme en vecteurs, puis entraîne un classificateur basé sur une régression logistique à partir de ces caractéristiques.

2. **[3 points]** Écrivez une fonction utilitaire qui retourne un vocabulaire de N‑grammes, *ngram_vocab*, pour toute valeur de *n* ≥ 1, étant donné un jeu de données *tokenized_reviews* étiquetées (une liste de tuples *(tokenized_review, review_class)*). Il est important que la taille du vocabulaire soit limitée à la taille maximale *max_vocab_size*. *ngram_vocab* doit inclure les *max_vocab_size* N‑grams les plus fréquents sur l’ensemble des critiques. Le type de retour doit être une liste de tuples, où chaque tuple représente un N‑gramme.

In [7]:
def get_ngram_tokenized(tokenized_review:list, n:int) -> list[tuple]:
    """
    Fonction qui construit un token de N-grammes à partir d'un token de mots.
    Args:
        tokenized_review : list 
            Token de mots d'une critique.
        n : int
            La taille des N-grammes à construire.
    """
    grams = [tokenized_review[i:] for i in range(n)]
    return list(zip(*grams))

In [8]:
def get_ngram_vocabulary_maxsize(tokenized_reviews:list[tuple[list[str], str]], n:int, max_vocab_size:int):
    ngram_vocab = []
    ## TO DO
    ### 1) Construire token N-grammes par critique
    ngrams_tokenized_reviews = []
    for review, _ in tokenized_reviews:
        ngrams_tokenized_reviews.extend(get_ngram_tokenized(review, n))
    ### 2) Faire le compte des N-grammes sur l'ensemble des critiques
    ngramsTokenizedReviewsCounter = Counter(ngrams_tokenized_reviews)
    ### 3) Le Top-N définit le nouveau vocabulaire.
    ngramsTokenizedReviewsCounter = ngramsTokenizedReviewsCounter.most_common(max_vocab_size)
    ngram_vocab = list(list(zip(*ngramsTokenizedReviewsCounter))[0])

    
    ##
    return ngram_vocab

3. **[4 points]** Écrivez une fonction utilitaire qui prend un vocabulaire de N‑grammes, *ngram_vocab*, ainsi qu’une critique tokenisée, *tokenized_review*, et qui retourne une représentation vectorisée des comptes de N‑grammes dans la critique. Le type de retour doit être un torch tensor de float, où chaque indice correspond à un N‑gramme dans *ngram_vocab* et où les valeurs correspondent à leurs comptes respectifs.

In [9]:
def get_vectorized_review(tokenized_review:list[str], ngram_vocab:list[tuple]):
    vectorized_review = torch.Tensor([0.0])
    ## TO DO
    ngram_size = len(ngram_vocab[0])
    ### Construire token N-grammes de la critique
    review_ngrams = get_ngram_tokenized(tokenized_review, ngram_size)
    ### Compter les N-grammes de la critique 
    review_ngramsCounter = Counter(review_ngrams)

    vectorized_review = torch.zeros(len(ngram_vocab), dtype=torch.float)
    for i, ngram in enumerate(ngram_vocab):
        if ngram in review_ngramsCounter:
            vectorized_review[i] = review_ngramsCounter[ngram]

    
    ##
    return vectorized_review

4. **[4 points]** Écrivez une classe *NgramReviewDataset* qui implémente un objet *torch Dataset*. Elle doit prendre comme input *tokenized_reviews* (une liste de tuples *(tokenized_review, review_class)*) ainsi qu’un vocabulaire de N‑grammes, *ngram_vocab*. Votre fonction d’initialisation doit vectoriser toutes les critiques, de sorte que *self.vectorized_reviews* soit un torch tensor (ou une matrice) de taille |tokenized_reviews| × |ngram_vocab|. Quant à *self.labels*, il doit s’agir d’un torch tensor de taille |tokenized_reviews| × |review_classes|, où chaque élément est un vecteur one‑hot tel que [1,0] correspond à la classe « neg » et [0,1] à la classe « pos ».

In [10]:
class NgramReviewDataset(Dataset):
    def __init__(self, tokenized_reviews:list[tuple[list[str], str]], ngram_vocab:list[tuple]):
        vectorized_reviews = [] 
        labels = [] 
        ## TO DO
        for review, label in tokenized_reviews:
            vectorized_reviews.append(get_vectorized_review(review, ngram_vocab))
            labels.append(0 if label == 'neg' else 1)
        
        ##
        self.vectorized_reviews = torch.stack(vectorized_reviews, dim=0)
        self.labels = F.one_hot(torch.Tensor(labels).long()).float()
        self.length = len(labels)

    def __getitem__(self, index):
        ## TO DO
        vectorized_review = self.vectorized_reviews[index]
        label = self.labels[index]       
        
        ##
        return vectorized_review, label, index

    def __len__(self):
        return self.length

5. **[2 points]** Écrivez une fonction *get_ngram_dataloader* qui prend en entrée *tokenized_reviews*, *ngram_vocab*, une taille de lot *batch_size* et un hyperparamètre *shuffle*. Elle doit utiliser la classe *NgramReviewDataset* définie précédemment et retourner un objet de type *torch.utils.data.DataLoader* qui tient compte des paramètres d’entrée de la fonction.

In [11]:
def get_ngram_dataloader(tokenized_reviews:list[tuple[list[str], str]], ngram_vocab:list[tuple], batch_size:int, shuffle:bool):
    ## TO DO
    data = NgramReviewDataset(tokenized_reviews, ngram_vocab)
    dataloader = DataLoader(data, batch_size=batch_size, shuffle=shuffle)

    
    ##
    return dataloader

#### Régression logistique avec un réseau neuronal à une seule couche linéaire

Voici notre classe de modèle de régression logistique (LR) que nous utiliserons comme base pour nos classificateurs N‑grammes LR.

In [12]:
class LogisticRegression(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        
    def forward(self, x):
        return self.linear(x)

6. **[4 points]** Complétez la fonction *train* que nous utiliserons à la fois pour notre classificateur N‑grammes LR et pour le classificateur basé sur un LSTM. Dans les deux cas, nous effectuons une régression logistique sur un ensemble de caractéristiques variables ; la fonction de perte utilisée dans votre fonction *train* doit donc être choisie en conséquence.

In [13]:
def train(model, dataloader, lr = 0.001, epochs = 5):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        train_loop = tqdm(dataloader, desc=f"Epoch {epoch+1}")
        for i, (batch_inputs, batch_labels, _) in enumerate(train_loop):
            train_loop.set_description(f"Epoch {epoch+1} | Batch {i}")
            ## TO DO
            ### Clear gradients from optimizer
            optimizer.zero_grad()

            ### Forward pass
            outputs = model(batch_inputs)

            ### Compute loss
            loss = nn.CrossEntropyLoss()(outputs, batch_labels.argmax(dim=1))

            ### Backward pass
            loss.backward()

            ### Update parameters
            optimizer.step()

            
            ##
    return model  

Entraînons maintenant trois classificateurs N‑grammes en utilisant respectivement des caractéristiques unigrammes, bigrammes et trigrammes.

In [14]:
def train_LR_ngram_classifier(train_dataset:list[tuple[str, str]], review_classes:list[str], n:int, batch_size = 32): 
    tokenized_train = [(preprocess(review), review_class) for review, review_class in  train_dataset]
    ngram_vocab = get_ngram_vocabulary_maxsize(tokenized_train, n, 30000)
    
    train_dataloader = get_ngram_dataloader(tokenized_train, ngram_vocab, batch_size, shuffle=True)

    model = LogisticRegression(len(ngram_vocab), len(review_classes))
    trained_model = train(model, train_dataloader)
    return ngram_vocab, trained_model

In [15]:
unigram_vocab, unigram_model = train_LR_ngram_classifier(train_dataset, review_classes, 1)
bigram_vocab, bigram_model = train_LR_ngram_classifier(train_dataset, review_classes, 2)
trigram_vocab, trigram_model = train_LR_ngram_classifier(train_dataset, review_classes, 3)

Epoch 1:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/50 [00:00<?, ?it/s]

...

Epoch 1:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/50 [00:00<?, ?it/s]

#### Évaluation de nos classificateurs LR
Voici la fonction d’évaluation pour nos classificateurs. Elle prend le *test_dataset* ainsi qu’un modèle de classification entraîné et retourne les prédictions de classe du modèle pour chaque critique dans les données de test. Elle peut être utilisée pour les deux types de classificateurs présentés dans ce devoir.

In [16]:
def eval(model, dataloader):
    ids = []
    labels = []
    predictions = []
    model.eval()
    with torch.no_grad():
        for batch_inputs, batch_labels, batch_ids in dataloader:
            x = batch_inputs.to(device)
            y_pred = model(x)
            labels += batch_labels.tolist()
            predictions += y_pred.tolist()
            if not isinstance(batch_ids, tuple):
                batch_ids.tolist()
            ids += batch_ids
            
    results = [{'review_id':idx, 'class_label':label, 'prediction': pred} for idx, label, pred in zip(ids, labels, predictions)]
    return results

Évaluons maintenant nos classificateurs N‑grammes sur les données de test.

In [17]:
def eval_LR_ngram_classifier(test_dataset:list[tuple[str, str]], ngram_vocab:list[tuple], model):
    tokenized_test = [(preprocess(review), review_class) for review, review_class in  test_dataset]
    test_dataloader = get_ngram_dataloader(tokenized_test, ngram_vocab, batch_size=32, shuffle=False)
    results = eval(model, test_dataloader)
    return results

In [18]:
unigram_results = eval_LR_ngram_classifier(test_dataset, unigram_vocab, unigram_model)
bigram_results = eval_LR_ngram_classifier(test_dataset, bigram_vocab, bigram_model)
trigram_results = eval_LR_ngram_classifier(test_dataset, trigram_vocab, trigram_model)

#### Résultats

Enfin, comparons les performances relatives de nos classificateurs LR unigramme, bigramme et trigramme.

7. **[4 points]** Écrivez une fonction qui retourne un dictionnaire contenant la précision, le rappel et le score F1 globaux pour toutes les classes, ainsi que l’exactitude (accuracy) globale d’un classificateur.

In [19]:
def get_accuracy_scores(results:list[dict]):
    accuracy = dict([('precision', 0.0),('recall', 0.0), ('f1', 0.0), ('accuracy', 0.0)])
    classes = {'pos':1, 'neg':0}
    ## TO DO
    TP = FP = TN = FN = 0
    for result in results:
        class_label = result['class_label']
        prediction = result['prediction']
        true_label = class_label.index(max(class_label))
        pred_label = prediction.index(max(prediction))
        if true_label == 1 and pred_label == 1:
            TP += 1
        elif true_label == 0 and pred_label == 1:
            FP += 1
        elif true_label == 0 and pred_label == 0:
            TN += 1
        elif true_label == 1 and pred_label == 0:
            FN += 1

    
    accuracy['accuracy'] = (TP + TN) / (TP + FP + TN + FN)
    accuracy['precision'] = TP / (TP + FP) 
    accuracy['recall'] = TP / (TP + FN) 
    accuracy['f1'] = 2 * TP / (2 * TP + FP + FN) 
    ##
    return accuracy

Comparons maintenant les resultats de nos trois classificateurs N-grammes.

In [20]:
accuracies = {'unigram':get_accuracy_scores(unigram_results),
                  'bigram':get_accuracy_scores(bigram_results),
                  'trigram':get_accuracy_scores(trigram_results)}

In [21]:
pd.DataFrame.from_dict(accuracies, orient='index')

,precision,recall,f1,accuracy
unigram,0.890547,0.895,0.892768,0.8925
bigram,0.895833,0.860,0.877551,0.8800
trigram,0.839080,0.730,0.780749,0.7950


## Classificateur 2 : Classificateur basé sur un LSTM

Nous allons entraîner un classificateur basé sur un LSTM en utilisant l’état final de sortie du LSTM comme caractéristiques pour la classification des critiques.

8. **[3 points]** Écrivez une fonction utilitaire qui retourne le vocabulaire de tokens, *vocab*, étant donné un jeu de données de *tokenized_reviews* étiquetées (une liste de tuples *(tokenized_review, review_class)*). Il est important que la taille du vocabulaire soit limitée à *max_vocab_size*. *vocab* doit inclure les *max_vocab_size* tokens uniques les plus fréquents sur l’ensemble des critiques. Le type de retour sera différent du problème 2 : cette fois, *vocab* doit être un dictionnaire où les clés sont les types de tokens et les valeurs sont leurs indices, ordonnés par fréquence. Notez que les deux premiers éléments du dictionnaire *vocab* sont les tokens spéciaux '\[PAD\]' pour le remplissage (padding) du texte et '\[UNK\]' pour les mots inconnus, qui ont respectivement les indices réservés 0 et 1. Ainsi, l’indice du token le plus fréquent doit commencer à 2.

In [22]:
def get_vocabulary_maxsize(tokenized_reviews:list[tuple[list[str], str]], max_vocab_size:int):
    vocab = dict([('[PAD]', 0), ('[UNK]',1)])
    ## TO DO
    ### Construire token de l'ensemble des mots de toutes les critiques
    tokensOfEachReview=list(zip(*tokenized_reviews))[0]
    tokens = [token for review in tokensOfEachReview for token in review]
    ### Faire le compte des mots sur l'ensemble des critiques
    tokensCounter = Counter(tokens)
    ### Le Top-N définit le nouveau vocabulaire.
    tokensCounter = tokensCounter.most_common(max_vocab_size)
    vocab.update({token: idx+2 for idx, (token, _) in enumerate(tokensCounter)})

    
    ##
    return vocab

9. **[3 points]** Écrivez une fonction utilitaire qui prend un dictionnaire de vocabulaire, *vocab*, et une critique tokenisée, *tokenized_review*, et qui retourne une *indexed_review*, où chaque token a été remplacé par son indice correspondant dans le dictionnaire *vocab*. Si un mot n’est pas présent dans le vocabulaire, il doit être remplacé par l’indice du token spécial '\[UNK\]'. Le type de retour doit être un torch tensor de int64.

In [23]:
def get_indexed_review(tokenized_review:list[str], vocab:dict[str,int]):
    indexed_review = torch.tensor([0], dtype=torch.long)
    ## TO DO
    tokenized_review_idx = [vocab.get(token, vocab['[UNK]']) for token in tokenized_review]
    indexed_review = torch.cat((indexed_review, torch.tensor(tokenized_review_idx, dtype=torch.long)), dim=0)
    
    
    ##
    return indexed_review

10. **[4 points]** Écrivez une classe *LSTMReviewDataset* qui implémente un objet *torch Dataset*. Elle doit prendre en entrée *tokenized_reviews* (une liste de tuples *(tokenized_review, review_class)*) ainsi qu’un dictionnaire de vocabulaire, *vocab*. Votre fonction d’initialisation doit transformer toutes les critiques tokenisées en séquences d’indices, de sorte que *self.indexed_reviews* soit une liste de torch tensor. Quant à *self.labels*, il doit également s’agir d’un torch tensor de taille |tokenized_reviews| × |review_classes|, où chaque élément est un vecteur one‑hot tel que [1,0] correspond à la classe « neg » et [0,1] à la classe « pos ».

In [24]:
class LSTMReviewDataset(Dataset):
    def __init__(self, tokenized_reviews:list[tuple[list[str], str]], vocab:dict[str,int]):
        indexed_reviews = [] 
        labels = [] 
        ## TO DO
        for tokenized_review, label in tokenized_reviews:
            indexed_reviews.append(get_indexed_review(tokenized_review, vocab))
            labels.append(0 if label == 'neg' else 1)
        
              
        ##
        self.indexed_reviews = indexed_reviews
        self.labels = F.one_hot(torch.Tensor(labels).long()).float()
        self.length = len(labels)

    def __getitem__(self, idx):
        ## TO DO
        indexed_review = self.indexed_reviews[idx]
        label = self.labels[idx]

        
        ##
        return indexed_review, label, idx

    def __len__(self):
        return self.length

11. **[3 points]** Écrivez une fonction *collate* personnalisée. Cette fonction sert à standardiser tous les éléments d’un lot. Elle peut être passée au *dataloader* comme paramètre. Comme chaque élément est une séquence d’indices représentant une critique, les séquences peuvent avoir des longueurs différentes. Afin de les regrouper dans un seul tenseur, nous devons standardiser leurs longueurs pour que toutes les séquences d’un lot aient la même longueur, égale à celle de la séquence la plus longue du lot. Nous ferons cela en remplissant les séquences avec des zéros — l’indice du token spécial '\[PAD\]' — au début de chaque séquence. Par exemple, si la longueur maximale d’un lot est 5 et que nous avons la séquence [22,300,6584], elle deviendra [0,0,22,300,6584]. Les sorties finales *batched_sequences* et *batched_labels* doivent être des torch tensor de type *long*, avec les dimensions |batch_size| × |taille maximale de séquence| et |batch_size| × |output_dim|, où *output_dim* correspond au nombre de classes. *batched_ids* vous est déjà fourni.

In [25]:
def collate_fn(items):
    sequences, labels, ids = list(zip(*items))
    batched_sequences = torch.tensor([0],dtype=torch.long)
    batched_labels = torch.tensor([0],dtype=torch.long)
    ##
    max_length = max([len(seq) for seq in sequences])
    padded_sequences = []
    for seq in sequences:
        nPads = max_length - len(seq)
        pads = torch.zeros(nPads, dtype=torch.long)
        padded_seq = torch.cat((pads, seq), dim=0)
        padded_sequences.append(padded_seq)
    batched_sequences = torch.stack(padded_sequences, dim=0)
    batched_labels = torch.stack(labels, dim=0)

    
    ##
    return batched_sequences, batched_labels, ids  

12. **[2 points]** Écrivez une fonction *get_lstm_dataloader* qui prend en entrée *tokenized_reviews*, *vocab*, une taille de lot *batch_size*, un hyperparamètre *shuffle* et une fonction de *collate*. Elle doit utiliser la classe *LSTMReviewDataset* définie précédemment et retourner un objet de type *torch.utils.data.DataLoader* tenant compte des paramètres d’entrée.

In [26]:
def get_lstm_dataloader(tokenized_reviews:list[tuple[list[str], str]], vocab:dict[str,int], batch_size:int, shuffle:bool, collate_fn=collate_fn):
    ## TO DO
    data = LSTMReviewDataset(tokenized_reviews, vocab)
    dataloader = DataLoader(data, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)

    
    ##
    return dataloader

13. **[3 points]** Complétez la fonction *forward* de notre classificateur LSTM. Elle doit utiliser une boucle récurrente avec une cellule LSTM afin de produire des caractéristiques de classification basées sur le LSTM. N’utilisez que l’état caché final comme caractéristiques d’entrée pour la *final_classifier_layer*, qui sera un classificateur de régression logistique.

In [27]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, output_dim):
        super(LSTMClassifier, self).__init__()
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.embeddings = nn.Embedding(self.vocab_size, self.emb_dim, padding_idx=0)
        self.lstm = nn.LSTMCell(self.emb_dim, self.hidden_dim)
        self.final_classifier_layer = LogisticRegression(self.hidden_dim, self.output_dim)

    def forward(self, input): # |batch_size| x |sequence_length| x |emb_dim|
        batch_size = input.size()[0]
        sequence_length = input.size()[1]
        #our initial h_(i-1) and c_(i-1) vectors are dummies
        h_prior = torch.zeros(batch_size, self.hidden_dim)
        c_prior = torch.zeros(batch_size, self.hidden_dim)
        ## TO DO
        input_t = torch.transpose(input, 0, 1) # |sequence_length| x |batch_size| x |emb_dim|
        for i in range(sequence_length):
            x_i = self.embeddings(input_t[i])
            h_prior, c_prior = self.lstm(x_i, (h_prior, c_prior))
        

         # Use the final hidden state for classification
        output = self.final_classifier_layer(h_prior)
        
        ##
        return output 

#### Entraînons notre classificateur basé sur un LSTM !

L’entraînement du modèle prendra un certain temps — prévoyez environ 15 minutes sur un ordinateur portable de gamme moyenne.

In [28]:
def train_LR_lstm_classifier(train_dataset:list[tuple[str, str]], review_classes:list[str], batch_size = 32): 
    tokenized_train = [(preprocess(review), review_class) for review, review_class in  train_dataset]
    vocab = get_vocabulary_maxsize(tokenized_train, 10000)
    
    train_dataloader = get_lstm_dataloader(tokenized_train, vocab, batch_size, shuffle=True)

    model = LSTMClassifier(len(vocab), 64, 64, len(review_classes))
    trained_model = train(model, train_dataloader)
    return vocab, trained_model

In [29]:
lstm_vocab, lstm_model = train_LR_lstm_classifier(train_dataset, review_classes)

Epoch 1:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/50 [00:00<?, ?it/s]

Nous pouvons évaluer notre classificateur basé sur un LSTM à l’aide de la même fonction *eval()* définie précédemment. Nous l’évaluerons à la fois sur les données de test et d’entraînement afin de comparer les performances du modèle.

In [30]:
def eval_LR_lstm_classifier(test_dataset:list[tuple[str, str]], lstm_vocab:list[tuple], model, collate_fn=collate_fn):
    tokenized_test = [(preprocess(review), review_class) for review, review_class in  test_dataset]
    test_dataloader = get_lstm_dataloader(tokenized_test, lstm_vocab, batch_size=32, shuffle=False, collate_fn=collate_fn)
    results = eval(model, test_dataloader)
    return results

In [31]:
lstm_test_results = eval_LR_lstm_classifier(test_dataset, lstm_vocab, lstm_model)
lstm_train_results = eval_LR_lstm_classifier(train_dataset, lstm_vocab, lstm_model)
lstm_accuracies = {'lstm test':get_accuracy_scores(lstm_test_results),
                  'lstm train':get_accuracy_scores(lstm_train_results)}
pd.DataFrame.from_dict(lstm_accuracies, orient='index')

,precision,recall,f1,accuracy
lstm test,0.623529,0.53000,0.572973,0.60500
lstm train,0.911517,0.81125,0.858466,0.86625


## Questions bonus

14. **[1 point bonus]** Que remarquez-vous concernant la relation entre la précision, le rappel et le score F1 dans cette tâche ? Pourquoi cela pourrait-il être le cas ? Écrivez 1 ou 2 phrases expliquant votre observation.

On observe que les valeurs de ces métriques sont toutes relativement très proches les unes des autres dans les deux modèles. Celà suggère donc que les valeurs FN (False Negative) et FP(False Positif) sont quasi-identique. C'est le cas parce que les deux classes sont parfaitement équilibrées dans les données et elles ont le même poids dans la fonction de perte.

Les métriques ont des valeurs similaires.

15. **[1 point bonus]** Qu’observez-vous à propos des résultats du LSTM ? Selon vous, à quel point ce modèle serait-il performant pour classifier de nouvelles critiques du monde réel comparativement, par exemple, au classificateur bigramme ? Écrivez 2 à 3 phrases expliquant vos observations et votre hypothèse concernant la performance du modèle sur des données nouvelles et inédites.

Dans les résultats du LSTM on observe que les performances dans le jeu de test est très bas par rapport à celles dans le jeu d'entrainement. Celà suggère donc un surapprentissage du modèle.

Le modèle LSTM serait moins performant que le classificateur bigramme pour classifier de nouvelles critiques du monde. En effet, les valeurs des métriques du LSTM sur le jeu de test est très bas par rapport à celles obtenus avec le classificateur bigramme. Celà s'explique probablement par le fait que dans ce contexte le signal discriminant est essentiellement local et lexical. La mémoire à long terme n'apporte pas de l'information pertinente et rajoute du bruit.